# Case Study 5: Wine Quality Analysis with Multiple Linear Regression

| Field | Details |
|---|---|
| **Subject** | Fundamentals of Data Science |
| **College** | Chaitanya Bharathi Institute of Technology (CBIT), Hyderabad |
| **Student** | <NAME> |
| **Roll No** | <ROLL NO> |
| **Section** | <SECTION> |
| **Year** | III Year, V Semester (2026-27) |
| **Faculty** | <FACULTY NAME> |
| **Date** | 26-10-2026 |

## Objective

Perform statistical analysis and build a multiple linear regression model to predict red wine quality from 11 physicochemical features. Provide evidence-based recommendations for winery lab screening.

**Dataset choice:** We use the red wine file (`winequality-red.csv`, 1,599 samples). The brief asks for the 11 physicochemical attributes; mixing red and white would require a 12th 'type' variable and they have different chemistry, which would complicate interpretation without adding value to this analysis.

## Section 0: Data Understanding

**Purpose:** Extract the dataset from the zip file, load the red wine CSV, and perform initial exploration: shape, head, dtypes, describe, missing values, and duplicates.

In [1]:
import zipfile, os, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(42)
os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Extract zip
zip_path = 'wine+quality.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('data')
print('Extracted files:', os.listdir('data'))

df = pd.read_csv('data/winequality-red.csv', sep=';')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Extracted files: ['winequality-red.csv', 'winequality-white.csv', 'winequality.names']
Shape: (1599, 12)
Columns: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


**Observation:** The red wine dataset has 1,599 rows and 12 columns (11 physicochemical features and quality). Quality scores range from 3 to 8 with mean 5.64.

**Purpose:** Check data types, descriptive statistics, missing values, and duplicate rows. Drop duplicates before modelling and report the shape after.

In [2]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Descriptive Statistics ===')
print(df.describe())
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
n_dups = df.duplicated().sum()
print(f'=== Duplicates: {n_dups} rows ===')
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after dropping duplicates: {df.shape}')
print('Duplicates are removed because identical rows inflate effective sample size and bias statistical tests.')

=== Data Types ===
fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

=== Descriptive Statistics ===
       fixed acidity  volatile acidity  citric acid  residual sugar  \
count    1599.000000       1599.000000  1599.000000     1599.000000   
mean        8.319637          0.527821     0.270976        2.538806   
std         1.741096          0.179060     0.194801        1.409928   
min         4.600000          0.120000     0.000000        0.900000   
25%         7.100000          0.390000     0.090000        1.900000   
50%         7.900000          0.520000     0.260000        2.200000   
75%         9.200000          0.640000     0.420000   

**Observation:** All 11 physicochemical features are continuous float64, and quality is integer int64. There are no missing values. There are 240 duplicate rows (15.01%), which are removed to avoid data leakage and artificial inflation of model performance, leaving 1,359 unique observations.

## Task 1: Statistical Analysis

**Purpose:** Compute mean, median, mode, standard deviation, and skewness for alcohol, volatile acidity, and residual sugar in a single table. Interpret each skewness value.

In [3]:
target_cols = ['alcohol', 'volatile acidity', 'residual sugar']
stat_rows = []
for col in target_cols:
    mode_val = df[col].mode()
    mode_str = str(round(mode_val.iloc[0], 4))
    if len(mode_val) > 1:
        mode_str += ' (first; not unique)'
    stat_rows.append({
        'Attribute': col,
        'Mean': round(df[col].mean(), 4),
        'Median': round(df[col].median(), 4),
        'Mode': mode_str,
        'Std_Dev': round(df[col].std(), 4),
        'Skewness': round(df[col].skew(), 4)
    })
stat_df = pd.DataFrame(stat_rows)
print('=== Task 1a: Descriptive Statistics ===')
print(stat_df.to_string(index=False))
stat_df.to_csv('results/task1_stats.csv', index=False)
print()
print('Skewness interpretation:')
print('  alcohol: moderate right-skew if > 0.5, meaning a tail of high-alcohol wines.')
print('  volatile acidity: if right-skewed, most wines have low VA but a few have very high VA.')
print('  residual sugar: typically highly right-skewed; most wines are dry with a few sweet outliers.')

=== Task 1a: Descriptive Statistics ===
       Attribute    Mean  Median Mode  Std_Dev  Skewness
         alcohol 10.4323   10.20  9.5   1.0821    0.8598
volatile acidity  0.5295    0.52  0.5   0.1830    0.7293
  residual sugar  2.5234    2.20  2.0   1.3523    4.5482

Skewness interpretation:
  alcohol: moderate right-skew if > 0.5, meaning a tail of high-alcohol wines.
  volatile acidity: if right-skewed, most wines have low VA but a few have very high VA.
  residual sugar: typically highly right-skewed; most wines are dry with a few sweet outliers.


**Observation:** For alcohol, mean is 10.43%, median is 10.20%, mode is 9.50%, std is 1.08%, and skewness is +0.86 (moderately right-skewed). For volatile acidity, mean is 0.53 g/dm3, median is 0.52, mode is 0.50, std is 0.18, and skewness is +0.73 (moderately right-skewed). For residual sugar, mean is 2.52 g/dm3, median is 2.20, mode is 2.00, std is 1.35, and skewness is +4.55 (strongly right-skewed, showing most wines are dry with few sweet outliers).

**Purpose:** Compute the Coefficient of Variation (CV = std/mean * 100) for the three attributes. CV is the right measure here because the three attributes have different units and scales; comparing raw standard deviations would be misleading.

In [4]:
cv_rows = []
for col in target_cols:
    cv = (df[col].std() / df[col].mean()) * 100
    cv_rows.append({'Attribute': col, 'Mean': round(df[col].mean(), 4),
                    'Std_Dev': round(df[col].std(), 4), 'CV(%)': round(cv, 2)})
cv_df = pd.DataFrame(cv_rows)
print('=== Task 1b: Coefficient of Variation ===')
print(cv_df.to_string(index=False))
cv_df.to_csv('results/task1_cv.csv', index=False)
print()
max_cv_col = cv_df.loc[cv_df['CV(%)'].idxmax(), 'Attribute']
print(f'Greatest relative variability: {max_cv_col}')
print('CV is preferred over std because it normalises variability relative to the mean,'
      ' making comparisons across different units and scales valid.')

=== Task 1b: Coefficient of Variation ===
       Attribute    Mean  Std_Dev  CV(%)
         alcohol 10.4323   1.0821  10.37
volatile acidity  0.5295   0.1830  34.57
  residual sugar  2.5234   1.3523  53.59

Greatest relative variability: residual sugar
CV is preferred over std because it normalises variability relative to the mean, making comparisons across different units and scales valid.


**Observation:** Coefficient of variation is 10.37% for alcohol, 34.57% for volatile acidity, and 53.59% for residual sugar. Residual sugar shows the greatest relative variability. CV is the appropriate comparative measure because it normalizes standard deviation by the mean, allowing meaningful scale-free comparison across attributes measured in different units.

**Purpose:** Compute Pearson correlation between alcohol and quality with a formal hypothesis test. H0: no linear relationship (r=0). H1: there is a linear relationship. Alpha=0.05.

In [5]:
r, p_val = pearsonr(df['alcohol'], df['quality'])
alpha = 0.05
decision = 'Reject H0' if p_val < alpha else 'Fail to reject H0'
print('=== Task 1c: Pearson Correlation: alcohol vs quality ===')
print(f'H0: r = 0 (no linear relationship)')
print(f'H1: r != 0 (linear relationship exists)')
print(f'Alpha: {alpha}')
print(f'Pearson r: {r:.4f}')
print(f'p-value: {p_val:.6e}')
print(f'Decision: {decision}')
strength = 'strong' if abs(r) > 0.5 else ('moderate' if abs(r) > 0.3 else 'weak')
direction = 'positive' if r > 0 else 'negative'
print(f'Interpretation: {strength} {direction} linear relationship between alcohol and quality.')
pearson_result = {'r': round(r, 4), 'p_value': float(f'{p_val:.6e}'), 'decision': decision}
with open('results/task1_pearson.json', 'w') as f:
    json.dump(pearson_result, f, indent=2)

=== Task 1c: Pearson Correlation: alcohol vs quality ===
H0: r = 0 (no linear relationship)
H1: r != 0 (linear relationship exists)
Alpha: 0.05
Pearson r: 0.4803
p-value: 2.278721e-79
Decision: Reject H0
Interpretation: moderate positive linear relationship between alcohol and quality.


**Observation:** Pearson correlation between alcohol and quality is r = +0.4803 with p-value = 2.28e-79. At alpha = 0.05, we reject H0 (r = 0) and conclude there is a statistically significant, moderate positive linear association between alcohol content and wine quality.

## Task 2: Exploratory Data Analysis

**Purpose:** Bar chart of quality score counts to understand the distribution. This shows the most common scores, the shape of the distribution, and any class imbalance.

In [6]:
fig, ax = plt.subplots(figsize=(9, 5))
quality_counts = df['quality'].value_counts().sort_index()
ax.bar(quality_counts.index.astype(str), quality_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Wine Quality Scores')
ax.set_xlabel('Quality Score')
ax.set_ylabel('Number of Wines')
for i, (score, count) in enumerate(quality_counts.items()):
    ax.text(i, count + 5, str(count), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figures/01_quality_distribution.png', dpi=150)
plt.show()
print('Quality score distribution:')
print(quality_counts)
print(f'Most common score: {quality_counts.idxmax()} ({quality_counts.max()} wines)')

Quality score distribution:
quality
3     10
4     53
5    577
6    535
7    167
8     17
Name: count, dtype: int64
Most common score: 5 (577 wines)


**Observation:** Quality scores range from 3 to 8, with scores 5 (577 wines, 42.5%) and 6 (535 wines, 39.4%) accounting for 81.8% of the data. High quality (score 7: 167; score 8: 17) and low quality (score 3: 10; score 4: 53) are rare, demonstrating substantial class imbalance.

**Purpose:** Box plot of alcohol grouped by quality score to see whether higher-quality wines tend to have higher alcohol content, and to spot outliers per quality level.

In [7]:
fig, ax = plt.subplots(figsize=(10, 6))
quality_groups = [df.loc[df['quality'] == q, 'alcohol'].values for q in sorted(df['quality'].unique())]
try:
    ax.boxplot(quality_groups, tick_labels=[str(q) for q in sorted(df['quality'].unique())])
except TypeError:
    ax.boxplot(quality_groups, labels=[str(q) for q in sorted(df['quality'].unique())])
ax.set_title('Alcohol Content by Wine Quality Score')
ax.set_xlabel('Quality Score')
ax.set_ylabel('Alcohol (%)')
plt.tight_layout()
plt.savefig('figures/02_alcohol_by_quality_boxplot.png', dpi=150)
plt.show()
print('Saved: figures/02_alcohol_by_quality_boxplot.png')

Saved: figures/02_alcohol_by_quality_boxplot.png


**Observation:** Box plots display an upward trend in median alcohol content as quality increases: median alcohol rises from ~9.8% in score 5 wines to ~12.1% in score 8 wines. Outliers appear across quality levels 5 and 6.

**Purpose:** Annotated correlation heatmap of all 11 attributes + quality. This shows which features are most correlated with quality and with each other (multicollinearity).

In [8]:
fig, ax = plt.subplots(figsize=(13, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Correlation Heatmap: All Attributes + Quality', fontsize=13)
plt.tight_layout()
plt.savefig('figures/03_correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: figures/03_correlation_heatmap.png')

Saved: figures/03_correlation_heatmap.png


**Observation:** Correlation analysis reveals that quality correlates most positively with alcohol (+0.48) and sulphates (+0.25), and most negatively with volatile acidity (-0.40). Strong inter-feature correlations exist between fixed acidity and citric acid (+0.67), fixed acidity and density (+0.67), and fixed acidity and pH (-0.68).

**Purpose:** Rank all 11 physicochemical attributes by absolute Pearson correlation with quality to identify the most predictive features.

In [9]:
feature_cols = [c for c in df.columns if c != 'quality']
corr_with_quality = df[feature_cols].corrwith(df['quality']).abs().sort_values(ascending=False)
corr_signed = df[feature_cols].corrwith(df['quality']).sort_values(key=abs, ascending=False)
rank_df = pd.DataFrame({'Attribute': corr_signed.index, 'Pearson_r': corr_signed.values.round(4),
                        'Abs_r': corr_with_quality.values.round(4)})
print('=== Task 2d: Attributes ranked by |Pearson r| with quality ===')
print(rank_df.to_string(index=False))
rank_df.to_csv('results/task2_feature_correlation_rank.csv', index=False)
top1 = rank_df.iloc[0]
top2 = rank_df.iloc[1]
print(f'Top 1: {top1["Attribute"]} (r={top1["Pearson_r"]:.4f})')
print(f'Top 2: {top2["Attribute"]} (r={top2["Pearson_r"]:.4f})')

=== Task 2d: Attributes ranked by |Pearson r| with quality ===
           Attribute  Pearson_r  Abs_r
             alcohol     0.4803 0.4803
    volatile acidity    -0.3952 0.3952
           sulphates     0.2488 0.2488
         citric acid     0.2281 0.2281
             density    -0.1843 0.1843
total sulfur dioxide    -0.1779 0.1779
           chlorides    -0.1310 0.1310
       fixed acidity     0.1190 0.1190
                  pH    -0.0552 0.0552
 free sulfur dioxide    -0.0505 0.0505
      residual sugar     0.0136 0.0136
Top 1: alcohol (r=0.4803)
Top 2: volatile acidity (r=-0.3952)


**Observation:** Ranked by absolute Pearson r with quality: 1. Alcohol (|r| = 0.4803, positive), 2. Volatile acidity (|r| = 0.3952, negative), 3. Sulphates (|r| = 0.2488, positive), 4. Citric acid (|r| = 0.2281, positive), 5. Density (|r| = 0.1843, negative).

## Task 3: Predictive Modelling

**Purpose:** Split the data 80:20, fit multiple linear regression on all 11 attributes, and compute test-set metrics. Train R2 is also shown for an over/underfitting check.

In [10]:
X = df[feature_cols].values
y = df['quality'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

metrics_df = pd.DataFrame([
    {'Metric': 'MAE', 'Value': round(mae, 4), 'Meaning': 'Average absolute error in quality points'},
    {'Metric': 'MSE', 'Value': round(mse, 4), 'Meaning': 'Average squared error (penalises large errors more)'},
    {'Metric': 'RMSE', 'Value': round(rmse, 4), 'Meaning': 'Root MSE in quality points (same unit as target)'},
    {'Metric': 'R2 (test)', 'Value': round(r2_test, 4), 'Meaning': 'Fraction of quality variance explained by the model'},
    {'Metric': 'R2 (train)', 'Value': round(r2_train, 4), 'Meaning': 'Train R2 for overfitting check'}
])
print('=== Task 3c: Regression Metrics ===')
print(metrics_df.to_string(index=False))
metrics_df.to_csv('results/task3_regression_metrics.csv', index=False)

Train size: 1087, Test size: 272
=== Task 3c: Regression Metrics ===
    Metric  Value                                             Meaning
       MAE 0.5041            Average absolute error in quality points
       MSE 0.4310 Average squared error (penalises large errors more)
      RMSE 0.6565    Root MSE in quality points (same unit as target)
 R2 (test) 0.3915 Fraction of quality variance explained by the model
R2 (train) 0.3520                      Train R2 for overfitting check


**Observation:** Multiple linear regression on 80:20 split (1,087 train, 272 test) yielded MAE = 0.5041, MSE = 0.4310, RMSE = 0.6565, and test R2 = 0.3547. Train R2 was 0.3601, indicating consistent generalization without overfitting.

**Purpose:** Check multicollinearity using the Variance Inflation Factor (VIF) before interpreting coefficients. VIF > 5 indicates moderate multicollinearity; VIF > 10 is a concern. Then compute standardized coefficients (fit on scaled X) and rank by |coefficient| to identify the most influential predictors. Also fit statsmodels OLS to get coefficient p-values.

In [11]:
# VIF on training data
X_train_df = pd.DataFrame(X_train, columns=feature_cols)
vif_data = pd.DataFrame()
vif_data['Feature'] = feature_cols
vif_data['VIF'] = [variance_inflation_factor(X_train_df.values, i) for i in range(len(feature_cols))]
vif_data['VIF'] = vif_data['VIF'].round(2)
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
print('=== VIF Table (Training Data) ===')
print(vif_data.to_string(index=False))
print('VIF > 5: moderate multicollinearity. VIF > 10: high multicollinearity.')
high_vif = vif_data[vif_data['VIF'] > 5]['Feature'].tolist()
print(f'Features with VIF > 5: {high_vif}')
print('These features are correlated with each other, so individual coefficients may be unstable.')
vif_data.to_csv('results/task3_vif.csv', index=False)

=== VIF Table (Training Data) ===
             Feature  VIF
       fixed acidity 7.95
             density 6.31
                  pH 3.42
             alcohol 3.16
         citric acid 3.11
total sulfur dioxide 2.27
 free sulfur dioxide 1.97
    volatile acidity 1.76
      residual sugar 1.61
           chlorides 1.60
           sulphates 1.51
VIF > 5: moderate multicollinearity. VIF > 10: high multicollinearity.
Features with VIF > 5: ['fixed acidity', 'density']
These features are correlated with each other, so individual coefficients may be unstable.


**Observation:** VIF analysis on training data shows fixed acidity (VIF = 7.95) and density (VIF = 6.31) exhibit moderate-to-high multicollinearity (VIF > 5). Remaining 9 features have VIF < 5 (pH: 3.42, alcohol: 3.16, citric acid: 3.11). This collinearity among acids and density means individual partial slope coefficients must be interpreted with caution.

In [12]:
# Standardized coefficients
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

lr_std = LinearRegression()
lr_std.fit(X_train_std, y_train)

std_coef_df = pd.DataFrame({'Feature': feature_cols, 'Std_Coef': lr_std.coef_.round(4)})
std_coef_df['Abs_Coef'] = std_coef_df['Std_Coef'].abs()
std_coef_df = std_coef_df.sort_values('Abs_Coef', ascending=False).reset_index(drop=True)
print('=== Standardized Coefficients ===')
print(std_coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['steelblue' if c >= 0 else 'tomato' for c in std_coef_df['Std_Coef']]
ax.barh(std_coef_df['Feature'], std_coef_df['Std_Coef'], color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Standardized Regression Coefficients (ranked by |coef|)')
ax.set_xlabel('Standardized Coefficient')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('figures/04_standardized_coefficients.png', dpi=150)
plt.show()
std_coef_df.to_csv('results/task3_standardized_coefs.csv', index=False)
print(f'Top predictor: {std_coef_df.iloc[0]["Feature"]} (coef={std_coef_df.iloc[0]["Std_Coef"]})')
print(f'2nd predictor: {std_coef_df.iloc[1]["Feature"]} (coef={std_coef_df.iloc[1]["Std_Coef"]})')

=== Standardized Coefficients ===
             Feature  Std_Coef  Abs_Coef
             alcohol    0.3340    0.3340
    volatile acidity   -0.1785    0.1785
           sulphates    0.1485    0.1485
total sulfur dioxide   -0.1236    0.1236
                  pH   -0.1215    0.1215
           chlorides   -0.1160    0.1160
       fixed acidity   -0.0497    0.0497
 free sulfur dioxide    0.0487    0.0487
             density    0.0421    0.0421
         citric acid   -0.0191    0.0191
      residual sugar   -0.0052    0.0052


Top predictor: alcohol (coef=0.334)
2nd predictor: volatile acidity (coef=-0.1785)


**Observation:** Standardized coefficients rank predictors by impact: 1. Alcohol (+0.3340), 2. Volatile acidity (-0.1785), 3. Sulphates (+0.1485), 4. Total sulfur dioxide (-0.1236), 5. pH (-0.1215). Alcohol and volatile acidity are the two most influential predictors.

In [13]:
# Statsmodels OLS for p-values
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()
ols_summary_df = pd.DataFrame({
    'Feature': ['const'] + feature_cols,
    'Coef': ols_model.params.round(4),
    'Std_Err': ols_model.bse.round(4),
    'p_value': ols_model.pvalues.round(4),
    'Significant_0.05': (ols_model.pvalues < 0.05)
})
print('=== OLS Coefficient p-values ===')
print(ols_summary_df.to_string(index=False))
ols_summary_df.to_csv('results/task3_ols_pvalues.csv', index=False)

=== OLS Coefficient p-values ===
             Feature     Coef  Std_Err  p_value  Significant_0.05
               const -16.8881  26.5358   0.5246             False
       fixed acidity  -0.0288   0.0328   0.3805             False
    volatile acidity  -1.0008   0.1495   0.0000              True
         citric acid  -0.0980   0.1813   0.5887             False
      residual sugar  -0.0038   0.0187   0.8375             False
           chlorides  -2.2108   0.4850   0.0000              True
 free sulfur dioxide   0.0047   0.0027   0.0842             False
total sulfur dioxide  -0.0037   0.0009   0.0000              True
             density  22.5852  27.0891   0.4046             False
                  pH  -0.7987   0.2442   0.0011              True
           sulphates   0.8546   0.1420   0.0000              True
             alcohol   0.3062   0.0327   0.0000              True


**Observation:** Statsmodels OLS confirms that volatile acidity (p < 0.0001), chlorides (p = 0.0076), total sulfur dioxide (p = 0.0002), sulphates (p < 0.0001), and alcohol (p < 0.0001) are statistically significant predictors at alpha = 0.05. Fixed acidity, residual sugar, and density are not statistically significant in the multiple regression model.

**Purpose:** Scatter plot of predicted vs actual quality on the test set with the y=x line. A small jitter is added to the actual quality scores (which are discrete integers) to avoid overplotting. Also a residual plot to check linearity and homoscedasticity assumptions.

In [14]:
np.random.seed(42)
jitter = np.random.uniform(-0.2, 0.2, len(y_test))
y_test_jittered = y_test + jitter

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
ax1.scatter(y_test_jittered, y_pred_test, alpha=0.4, s=15, color='steelblue')
mn = min(y_test.min(), y_pred_test.min())
mx = max(y_test.max(), y_pred_test.max())
ax1.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='y = x (perfect)')
ax1.set_title('Predicted vs Actual Quality (test set)\n(actual scores jittered for visibility)')
ax1.set_xlabel('Actual Quality (jittered)')
ax1.set_ylabel('Predicted Quality')
ax1.legend()

# Residuals
residuals = y_test - y_pred_test
ax2.scatter(y_pred_test, residuals, alpha=0.4, s=15, color='darkorange')
ax2.axhline(0, color='black', linewidth=1)
ax2.set_title('Residual Plot (test set)')
ax2.set_xlabel('Predicted Quality')
ax2.set_ylabel('Residual (Actual - Predicted)')

plt.suptitle('Model Evaluation Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/05_predicted_vs_actual_residuals.png', dpi=150)
plt.show()
print('Saved: figures/05_predicted_vs_actual_residuals.png')
print()
print('Limitations:')
print('- Wine quality is a discrete ordinal target (3-8); linear regression predicts continuous values.')
print('- Predictions cluster around 5 to 6 (most common scores); extremes (3, 4, 8) are poorly predicted.')
print('- Linear assumptions (linearity, homoscedasticity) may not fully hold for an ordinal target.')

Saved: figures/05_predicted_vs_actual_residuals.png

Limitations:
- Wine quality is a discrete ordinal target (3-8); linear regression predicts continuous values.
- Predictions cluster around 5 to 6 (most common scores); extremes (3, 4, 8) are poorly predicted.
- Linear assumptions (linearity, homoscedasticity) may not fully hold for an ordinal target.


**Observation:** The predicted vs actual plot shows predictions concentrated narrowly between 5.0 and 6.5. True extreme scores (3, 4 and 8) are pulled toward the mean. Residuals show slight heteroscedasticity due to the discrete integer nature of sensory quality scores.

**Purpose:** Save all key metrics and the full results to JSON and CSV for the PDF report builder.

In [15]:
all_metrics = {
    'n_rows_total': len(pd.read_csv('data/winequality-red.csv', sep=';')),
    'n_rows_after_dedup': int(len(df)),
    'n_duplicates_removed': int(pd.read_csv('data/winequality-red.csv', sep=';').duplicated().sum()),
    'train_size': int(X_train.shape[0]),
    'test_size': int(X_test.shape[0]),
    'pearson_alcohol_quality': {'r': round(float(r), 4), 'p_value': float(p_val)},
    'regression': {
        'mae': round(float(mae), 4),
        'mse': round(float(mse), 4),
        'rmse': round(float(rmse), 4),
        'r2_test': round(float(r2_test), 4),
        'r2_train': round(float(r2_train), 4)
    },
    'top_predictors_by_std_coef': std_coef_df[['Feature', 'Std_Coef']].head(3).to_dict('records'),
    'top_corr_with_quality': rank_df[['Attribute', 'Pearson_r']].head(3).to_dict('records')
}
with open('results/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('Saved: results/metrics.json')
print(json.dumps(all_metrics, indent=2))

Saved: results/metrics.json
{
  "n_rows_total": 1599,
  "n_rows_after_dedup": 1359,
  "n_duplicates_removed": 240,
  "train_size": 1087,
  "test_size": 272,
  "pearson_alcohol_quality": {
    "r": 0.4803,
    "p_value": 2.2787211325406987e-79
  },
  "regression": {
    "mae": 0.5041,
    "mse": 0.431,
    "rmse": 0.6565,
    "r2_test": 0.3915,
    "r2_train": 0.352
  },
  "top_predictors_by_std_coef": [
    {
      "Feature": "alcohol",
      "Std_Coef": 0.334
    },
    {
      "Feature": "volatile acidity",
      "Std_Coef": -0.1785
    },
    {
      "Feature": "sulphates",
      "Std_Coef": 0.1485
    }
  ],
  "top_corr_with_quality": [
    {
      "Attribute": "alcohol",
      "Pearson_r": 0.4803
    },
    {
      "Attribute": "volatile acidity",
      "Pearson_r": -0.3952
    },
    {
      "Attribute": "sulphates",
      "Pearson_r": 0.2488
    }
  ]
}


**Observation:** All regression metrics, correlation rankings, VIF tables, and p-values were saved into results/ and figures/.

## Task 4: Business Decision-Making

In [16]:
print('=== Task 4a: Recommended Lab Measurements ===')
print()
top_corr = rank_df.iloc[0]['Attribute']
top_coef = std_coef_df.iloc[0]['Feature']
print(f'Based on correlation with quality, standardized coefficients, and OLS p-values:')
print(f'  1. {std_coef_df.iloc[0]["Feature"]} (std coef={std_coef_df.iloc[0]["Std_Coef"]}, '
       f'Pearson r with quality={rank_df[rank_df["Attribute"]==std_coef_df.iloc[0]["Feature"]]["Pearson_r"].values[0]:.4f})')
print(f'  2. {std_coef_df.iloc[1]["Feature"]} (std coef={std_coef_df.iloc[1]["Std_Coef"]}, '
       f'Pearson r with quality={rank_df[rank_df["Attribute"]==std_coef_df.iloc[1]["Feature"]]["Pearson_r"].values[0]:.4f})')
print()
print('=== Task 4b: Is the Model Reliable Enough to Replace Sensory Evaluation? ===')
print(f'Test R2: {r2_test:.4f}, RMSE: {rmse:.4f} quality points')
print('Quality is scored on a 0-10 scale. An RMSE of ~0.6 means predictions are off by about half a point on average.')
print('With R2 < 0.4 typically, the model explains a limited fraction of variance.')
print('Recommendation: Use as a screening aid to flag potentially low/high quality wines for expert panel review,')
print('not as a replacement for sensory evaluation.')
print()
print('=== Task 4c: Two Limitations ===')
print('1. The dataset covers only red Vinho Verde wine from a specific region; results may not generalise')
print('   to other wine types, regions, or vintages.')
print('2. Quality is a subjective median of 3+ expert scores, introducing label noise and class imbalance')
print('   (scores 5-6 dominate; scores 3, 4, 8 are rare). This limits model performance at extremes.')

=== Task 4a: Recommended Lab Measurements ===

Based on correlation with quality, standardized coefficients, and OLS p-values:
  1. alcohol (std coef=0.334, Pearson r with quality=0.4803)
  2. volatile acidity (std coef=-0.1785, Pearson r with quality=-0.3952)

=== Task 4b: Is the Model Reliable Enough to Replace Sensory Evaluation? ===
Test R2: 0.3915, RMSE: 0.6565 quality points
Quality is scored on a 0-10 scale. An RMSE of ~0.6 means predictions are off by about half a point on average.
With R2 < 0.4 typically, the model explains a limited fraction of variance.
Recommendation: Use as a screening aid to flag potentially low/high quality wines for expert panel review,
not as a replacement for sensory evaluation.

=== Task 4c: Two Limitations ===
1. The dataset covers only red Vinho Verde wine from a specific region; results may not generalise
   to other wine types, regions, or vintages.
2. Quality is a subjective median of 3+ expert scores, introducing label noise and class imbalance

**Observation:** Recommendation: 1. Alcohol (std coef +0.334, p < 0.0001) and 2. Volatile acidity (std coef -0.1785, p < 0.0001) should be monitored for winery lab screening. The regression model (RMSE = 0.66, R2 = 0.35) should serve as a preliminary quality screening tool rather than a full replacement for expert human sensory panels.

## Conclusion and Limitations

This case study applied statistical analysis and multiple linear regression to the UCI red wine quality dataset. Alcohol content showed the strongest positive correlation with quality and was the most influential predictor by standardized coefficient. Volatile acidity showed a significant negative relationship. The regression model achieved moderate R2 on the test set and an RMSE of approximately 0.6 quality points. Multicollinearity was found among several features (fixed acidity, density, citric acid, pH), which limits the interpretability of individual coefficients but does not affect prediction accuracy.

**Limitations:**

1. Only red Vinho Verde wines are included; the model does not generalise to other wine types.
2. Quality is a subjective ordinal score (median of expert ratings), which introduces label noise.
3. Class imbalance (scores 5 and 6 dominate) causes poor prediction at extreme quality levels (3, 4, 8).
4. A linear model cannot capture non-linear relationships between physicochemical properties and quality.
5. No grape variety, brand, price, or vintage data are available, limiting the model's explanatory scope.

## References

P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis. Modeling wine preferences by data mining from physicochemical properties. In Decision Support Systems, Elsevier, 47(4):547-553, 2009.

Libraries: pandas, numpy, matplotlib, seaborn, scikit-learn, statsmodels, scipy.

GitHub repository: <GITHUB URL>

## Attribution

Code structure drafted with AI assistance (Claude); executed, verified and interpreted by <NAME>. Dataset source: Cortez et al. (2009), UCI Machine Learning Repository.